# Evaluation of both methods on test set

In [21]:
import json
import pandas as pd
from collections import Counter

from src.skills import TIER1_PHRASES, TIER2_EXACT_CASE, TIER3_CUSTOM
from src.models import baseline_extract_skills, build_nlp, entity_ruler_extract_skills
from src.evaluation import (
    build_alias_to_canonical, 
    normalize_all_ads, 
    calculate_micro_metrics, 
    calculate_macro_metrics, 
    create_error_analysis,
    summarize_errors,
    errors_to_dataframe,
    build_canonical_to_aliases,
    review_ad,
)

In [22]:
with open("../data/labeled_test_set.json", "r", encoding="utf-8") as file:
    test_set = json.load(file)

test_df = pd.DataFrame(test_set)
test_df.head(3)

,id,description,skills,spans
0,31191172,Vi söker nu en erfaren kravanalytiker till ett...,"[Analysera verksamhetsprocesser, Certifiering ...","[[438, 464, IT-relaterad kravställning], [525,..."
1,30929936,Om företaget Avaron hjälper dig att hitta uppd...,"[Architecture Design Document, TOGAF, TOGAF-ce...","[[759, 773, lösningsdesign], [1123, 1140, arki..."
2,30895651,IT-avdelningen på Stockholms universitet bestå...,"[API-hantering, Active Directory, Analysera kr...","[[844, 899, designa, vidareutveckla och kvalit..."


In [23]:
test_df = test_df.drop(columns="spans")
test_df = test_df.rename(columns={"skills": "gold_raw"}) 

In [24]:
alias_to_canonical = build_alias_to_canonical(TIER1_PHRASES, TIER2_EXACT_CASE, TIER3_CUSTOM)

gold_raw = dict(zip(test_df["id"], test_df["gold_raw"]))
gold = normalize_all_ads(gold_raw, alias_to_canonical)

## Baseline

In [25]:
with open("../data/taxonomi_2511.json", "r", encoding="utf-8") as f:
    skills_df = pd.DataFrame(json.load(f))

skills_short = (
    skills_df["skill"]
    .str.split(",").str[0]
    .str.strip()
    .drop_duplicates()
    .tolist()
)

test_df["baseline_pred_raw"] = test_df["description"].apply(
    lambda text: baseline_extract_skills(text, skills_short)
)
test_df.head(3)

,id,description,gold_raw,baseline_pred_raw
0,31191172,Vi söker nu en erfaren kravanalytiker till ett...,"[Analysera verksamhetsprocesser, Certifiering ...",[]
1,30929936,Om företaget Avaron hjälper dig att hitta uppd...,"[Architecture Design Document, TOGAF, TOGAF-ce...",[]
2,30895651,IT-avdelningen på Stockholms universitet bestå...,"[API-hantering, Active Directory, Analysera kr...",[]


In [26]:
baseline_pred_raw = dict(zip(test_df["id"], test_df["baseline_pred_raw"]))

# convert all ads, for gold and predictions
baseline_pred = normalize_all_ads(baseline_pred_raw, alias_to_canonical)

In [27]:
baseline_micro_metrics = calculate_micro_metrics(gold, baseline_pred)

baseline_micro_metrics

{'precision': 0.5,
 'recall': 0.021,
 'f1': 0.039,
 'tp': 31,
 'fp': 31,
 'fn': 1480}

## Entity Ruler

In [28]:
# Building pipeline
nlp = build_nlp()

ruler = nlp.get_pipe("entity_ruler")

# Show how many patterns
print(len(ruler.patterns), "patterns")

852 patterns


In [29]:
test_df["ruler_pred_raw"] = test_df["description"].apply(lambda text:entity_ruler_extract_skills(text, nlp))

test_df.head()

,id,description,gold_raw,baseline_pred_raw,ruler_pred_raw
0,31191172,Vi söker nu en erfaren kravanalytiker till ett...,"[Analysera verksamhetsprocesser, Certifiering ...",[],"[Driva workshops, IREB, IT-miljöer, Integratio..."
1,30929936,Om företaget Avaron hjälper dig att hitta uppd...,"[Architecture Design Document, TOGAF, TOGAF-ce...",[],"[Digitalisering, IT-miljöer, Integrationer, Sw..."
2,30895651,IT-avdelningen på Stockholms universitet bestå...,"[API-hantering, Active Directory, Analysera kr...",[],"[Active Directory, English, Entra ID, IT-miljö..."
3,31381155,Vill du arbeta med arkitekturen bakom nästa ge...,"[AIDL, AIDL/HIDL, ASPICE L3, Android 13, Andro...",[Android],"[C++, Digitalisering, Local Interconnect Netwo..."
4,31543038,Vill du arbeta i skärningspunkten mellan verks...,"[AI, AI-agenter, AI-assisterade beslutsstöd, A...",[],"[Artificial Intelligence, Automation, Business..."


In [30]:
ruler_pred_raw = dict(zip(test_df["id"], test_df["ruler_pred_raw"]))

# convert all ads, for gold and predictions
ruler_pred = normalize_all_ads(ruler_pred_raw, alias_to_canonical)

In [31]:
ruler_micro_metrics = calculate_micro_metrics(gold, ruler_pred)

ruler_micro_metrics

{'precision': 0.692,
 'recall': 0.187,
 'f1': 0.295,
 'tp': 283,
 'fp': 126,
 'fn': 1228}

### Comparison

In [32]:
comparison = pd.DataFrame({"baseline": baseline_micro_metrics, "ruler": ruler_micro_metrics})

comparison.head(3)

,baseline,ruler
precision,0.500,0.692
recall,0.021,0.187
f1,0.039,0.295


## Error analysis for EntityRuler

In [ ]:
errors = create_error_analysis(gold, ruler_pred)
summary = summarize_errors(errors, alias_to_canonical)

print("\nMost common missed skills:\n", summary["top_fn"])
print("\nMost common mistakes:\n", summary["top_fp"])

In [35]:
errs = errors_to_dataframe(errors)
errs.head(10) 

,id,false_positives,false_negatives,n_fp,n_fn
0,31535195,"[Change management, SAP]","[ALE, ANSI X12, AS2, B2B Integrator, B2B Integ...",2,53
1,31381155,"[Digitalisering, User experience]","[AIDL, AIDL/HIDL, ASPICE L3, Android 13, Andro...",2,47
2,30448499,"[Automation, Nätverksteknik]","[AIOps, AIOps‑lösningar, AIOps‑plattformar, Az...",2,41
3,31531216,[Swedish],"[INCOSE, Integration of systems and subsystems...",1,36
4,31366889,"[IT-miljöer, Information Architecture]","[IEC 61850, IP-baserad kommunikation, IT- och ...",2,36
5,31530871,"[General Data Protection Regulation (GDPR), IT...","[API- och integrationsarkitektur, Analysera in...",2,35
6,31543038,"[Driva workshops, Verksamhetsutveckling]","[AI-agenter, AI-assisterade beslutsstöd, AI-ba...",2,34
7,31030559,"[Cloud technology, Nätverksteknik]","[ARM, AWS Solutions Architect Professional, Az...",2,34
8,31024157,[DevOps],"[AI assistants, AI implementation, AI use case...",1,32
9,30895651,[Identity and Access Management (IAM)],"[API-hantering, Analysera krav och behovsbilde...",1,30


In [36]:
canonical_to_aliases = build_canonical_to_aliases(alias_to_canonical)
review_ad(31531216, test_df, gold, ruler_pred, canonical_to_aliases)

gold: ['INCOSE', 'Integration of systems and subsystems', 'Laboratory testing', 'RF', 'Requirements engineering', 'Software Defined Radio (SDR)', 'System design', 'System engineering', 'System integreration', 'Systems Engineering', 'Technical project management', 'Technical review of solutions and documentation', 'Test automation', 'Testing', 'Verification & Validation', 'communication links', 'complex technical systems', 'ground infrastructure', 'ground systems', 'integration, testing and verification', 'management systems', 'modems', 'onboard systems', 'radio and data communication systems', 'radio communications', 'radio systems', 'requirements decomposition', 'satellite communications', 'satellites', 'signal processing', 'space- and ground-based systems', 'system analysis', 'system architecture', 'system design', 'system design, requirements, integration and verification', 'system designs', 'technical subsystems', 'terminals', 'verification and validation activities']
pred: ['Swedi